# Limpar e Processar Dados

Template para limpeza de dados e push para o sistema MDM.

**Workflow:**
1. Carregar dados raw
2. Definir pipeline de limpeza
3. Rodar pipeline
4. Inspecionar resultado
5. Validar contra contrato Silver
6. Exportar ou push para API

In [ ]:
import datawork
datawork.setup()

## 1. Carregar Dados Raw

In [ ]:
from datawork.io.loaders import load_csv

# Opção A: carregar de CSV local
df_raw = load_csv("../data/NOME_DO_ARQUIVO.csv")

# Opção B: carregar da API (fonte já submetida)
# from datawork.io.loaders import load_raw_registros
# df_raw = load_raw_registros("FONTE_ID_AQUI")

print(f"Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} cols")
print(f"Colunas: {list(df_raw.columns)}")
df_raw.head(3)

## 2. Definir Pipeline

Montar o pipeline de limpeza usando stages composáveis. Ajustar os stages conforme a planilha.

In [ ]:
from datawork.pipeline.runner import PipelineRunner
from datawork.pipeline.stages import (
    drop_empty_rows,
    parse_sections,
    normalize_areas,
    normalize_addresses,
    extract_obs,
    classify_values,
    extract_contacts,
    compute_dedup_hash,
    drop_duplicates_by_hash,
)

# Para planilhas no formato Petrus, importar configs:
from petrus.infrastructure.mdm.connectors.petrus_config import (
    SECTION_MARKERS,
    STREET_CANONICAL,
    STREET_INFO,
    REGION_CITY_MAP,
)

pipeline = (
    PipelineRunner("limpeza")
    .add("sections", lambda df: parse_sections(df, SECTION_MARKERS))
    .add("empty", drop_empty_rows)
    .add("areas", normalize_areas)
    .add("addresses", lambda df: normalize_addresses(df, STREET_CANONICAL, STREET_INFO, REGION_CITY_MAP))
    .add("observations", extract_obs)
    .add("values", classify_values)
    .add("contacts", extract_contacts)
    .add("dedup", compute_dedup_hash)
    .add("dedup_drop", drop_duplicates_by_hash)
)

## 3. Rodar Pipeline

In [ ]:
df_clean = pipeline.run(df_raw)

In [ ]:
pipeline.summary()

## 4. Inspecionar Resultado

In [ ]:
from datawork.display import show_sample, show_stats

show_sample(df_clean, 10, "Amostra dos dados limpos")

In [ ]:
show_stats(df_clean)

## 5. Validar contra Contrato Silver

In [ ]:
from datawork.contracts.silver import CleanRecordSchema

try:
    df_validated = CleanRecordSchema.validate(df_clean, lazy=True)
    print("Validação Silver OK!")
except Exception as e:
    print("Erros de validação:")
    print(e)

## 6. Comparar com Dados Anteriores (opcional)

In [ ]:
from datawork.display import show_diff

# Carregar dados anteriores para comparação
# df_anterior = load_csv("../data/dados_anteriores.csv")
# show_diff(df_anterior, df_clean)

## 7. Exportar ou Push para API

In [ ]:
from datawork.io.pushers import export_csv, push_clean_to_api

# Opção A: salvar CSV limpo
export_csv(df_clean, "../output/dados_limpos.csv")

# Opção B: push direto para API
# push_clean_to_api(df_clean, fonte_id="FONTE_ID_AQUI")